# 토크나이저 비교 평가 (v1 자작 ↔ v2 HF)

같은 데이터·모델·파이프라인에서 **토크나이저만 다른** 두 모델을 평가셋(질문+정답)으로 비교한다.

- **bits-per-char(bpc)**: 정답에 대한 모델 확신도를 *글자 수*로 정규화 → 토크나이저가 달라도 비교 가능 (ppl은 비교 불가)
- **생성 + 정답 포함**: 답변을 N회 생성해 정답 문자열 포함 여부(거친 자동 채점) + 정성 확인

> 선행: 학습 노트북이 Drive 에 아래 4개를 저장해 둬야 함
> `tokenizer.json`(자작) · `tokenizer_hf.json`(HF) · `finetune_checkpoint_custom.pt`(v1) · `finetune_checkpoint.pt`(v2)

## 0. 환경

In [ ]:
!pip install tokenizers -q

In [ ]:
import os, sys, shutil

REPO_URL  = "https://github.com/kkkk2058/korean-chatbot"
REPO_DIR  = "korean-chatbot"
STAGE_DIR = f"{REPO_DIR}/stage1_from_scratch"
DRIVE_DIR = "/content/drive/MyDrive/korean_chatbot"

from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    !git -C $REPO_DIR pull

sys.path.insert(0, STAGE_DIR)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("준비 완료:", device)

## 1. 평가셋 (질문 + 정답 12문항)

In [ ]:
EVAL = [
    {"q": "한국의 수도는 어디인가요?",        "a": "서울",            "cat": "사실"},
    {"q": "물은 어떤 원소로 이루어져 있나요?",  "a": "수소와 산소",      "cat": "사실"},
    {"q": "태양계에서 가장 큰 행성은 무엇인가요?", "a": "목성",          "cat": "사실"},
    {"q": "한글을 만든 사람은 누구인가요?",     "a": "세종대왕",        "cat": "사실"},
    {"q": "1년은 며칠인가요?",               "a": "365일",          "cat": "사실"},
    {"q": "광합성이란 무엇인가요?",           "a": "식물이 빛으로 양분과 산소를 만드는 과정", "cat": "정의"},
    {"q": "중력이란 무엇인가요?",             "a": "물체끼리 서로 끌어당기는 힘", "cat": "정의"},
    {"q": "얼음이 녹으면 무엇이 되나요?",      "a": "물",             "cat": "추론"},
    {"q": "비가 올 때 무엇을 챙기면 좋을까요?", "a": "우산",           "cat": "추론"},
    {"q": "2 더하기 3은 얼마인가요?",         "a": "5",              "cat": "계산"},
    {"q": "안녕하세요?",                    "a": "안녕하세요",       "cat": "대화"},
    {"q": "고마워",                        "a": "천만에요",         "cat": "대화"},
]
print(f"평가셋 {len(EVAL)}문항")

## 2. 모델 + 토크나이저 로드 (v1 자작 / v2 HF)

In [ ]:
import math
import torch.nn.functional as F
from src.model import Transformer


def load_model(ckpt_path, n_heads=8):
    """체크포인트 shape 에서 아키텍처를 추론해 모델 로드."""
    ck = torch.load(ckpt_path, map_location=device)
    state = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    vocab, d_model = state["embedding.weight"].shape
    max_seq  = state["positional.weight"].shape[0]
    n_layers = 1 + max(int(k.split(".")[1]) for k in state if k.startswith("layers."))
    m = Transformer(vocab_size=vocab, d_model=d_model, n_heads=n_heads,
                    n_layers=n_layers, max_seq_len=max_seq, dropout=0.0).to(device)
    m.load_state_dict(state); m.eval()
    print(f"  로드 완료: vocab={vocab} d_model={d_model} layers={n_layers}")
    return m


# v1: 자작 순수 Python BPE
from src.tokenizer import BPETokenizer as CustomBPE
tok_v1 = CustomBPE()
tok_v1.load(f"{DRIVE_DIR}/tokenizer.json")

# v2: HuggingFace ByteLevel BPE (어댑터로 .vocab/.encode/.decode 통일)
from tokenizers import Tokenizer
class HFTok:
    def __init__(self, path):
        self.tk = Tokenizer.from_file(path)
        self.vocab = self.tk.get_vocab()
    def encode(self, text): return self.tk.encode(text).ids
    def decode(self, ids):  return self.tk.decode(ids)
tok_v2 = HFTok(f"{DRIVE_DIR}/tokenizer_hf.json")

print("v1(자작) 모델"); model_v1 = load_model(f"{DRIVE_DIR}/finetune_checkpoint_custom.pt")
print("v2(HF)   모델"); model_v2 = load_model(f"{DRIVE_DIR}/finetune_checkpoint.pt")
print(f"vocab: v1={len(tok_v1.vocab)}  v2={len(tok_v2.vocab)}")

## 3. 평가기 (bpc + 생성)

In [ ]:
class Evaluator:
    def __init__(self, name, model, tokenizer):
        self.name = name
        self.m    = model.eval()
        self.tk   = tokenizer
        self.bos  = tokenizer.vocab["<s>"]
        self.eos  = tokenizer.vocab["</s>"]
        self.dev  = next(model.parameters()).device
        self.max_seq = model.positional.num_embeddings

    @torch.no_grad()
    def answer_bpc(self, q, a):
        """정답 a 에 대한 글자당 손실(bits/char). 글자수로 정규화 → 토크나이저 독립 비교."""
        prompt = f"질문: {q}\n답변:"
        p_ids = [self.bos] + self.tk.encode(prompt)
        a_ids = self.tk.encode(" " + a)
        if not a_ids:
            return float("nan")
        ids    = torch.tensor([p_ids + a_ids], device=self.dev)
        logits = self.m(ids)[:, :-1, :]
        target = ids[:, 1:]
        nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                              target.reshape(-1), reduction="none")
        ans_nll = nll[len(p_ids) - 1:].sum().item()   # 정답 구간 nats 합
        return ans_nll / (len(a) * math.log(2))        # ÷ 글자수 ÷ ln2

    @torch.no_grad()
    def generate(self, q, max_new_tokens=100, temperature=0.8, top_k=30, rep=1.3):
        ids = [self.bos] + self.tk.encode(f"질문: {q}\n답변:")
        x = torch.tensor([ids], device=self.dev)
        for _ in range(max_new_tokens):
            if x.size(1) >= self.max_seq:
                break
            logits = self.m(x)[:, -1, :]
            for t in set(x[0].tolist()):
                logits[0, t] /= rep
            logits = logits / temperature
            k = min(top_k, logits.size(-1))
            v, _ = torch.topk(logits, k)
            logits[logits < v[:, -1:]] = float("-inf")
            nid = torch.multinomial(F.softmax(logits, dim=-1), 1)
            if nid.item() == self.eos:
                break
            x = torch.cat([x, nid], dim=1)
        return self.tk.decode(x[0].tolist()[len(ids):]).strip()


v1 = Evaluator("v1(자작)", model_v1, tok_v1)
v2 = Evaluator("v2(HF)",   model_v2, tok_v2)
print("평가기 준비 완료")

## 4. 실행 & 비교

In [ ]:
import statistics as st

def run_eval(ev, n_runs=3):
    rows = []
    for it in EVAL:
        bpc  = ev.answer_bpc(it["q"], it["a"])
        gens = [ev.generate(it["q"]) for _ in range(n_runs)]
        hit  = any(it["a"] in g for g in gens)   # 거친 자동 채점(정답 문자열 포함)
        rows.append({**it, "bpc": bpc, "hit": hit, "samples": gens})
    return rows

r1 = run_eval(v1)
r2 = run_eval(v2)

def summary(rows, name):
    print(f"== {name} ==")
    print(f"  평균 bpc        : {st.mean(r['bpc'] for r in rows):.3f}  (낮을수록 좋음)")
    print(f"  정답 포함(자동) : {sum(r['hit'] for r in rows)}/{len(rows)}")
    cats = {}
    for r in rows: cats.setdefault(r["cat"], []).append(r["bpc"])
    for c, vs in cats.items():
        print(f"    [{c}] 평균 bpc {st.mean(vs):.3f}")

summary(r1, "v1(자작)"); print(); summary(r2, "v2(HF)")

### 문항별 표 + 정성 샘플 (사람 채점용)

In [ ]:
# 문항별 bpc / 정답포함 비교
print(f"{'질문':<24}{'v1 bpc':>9}{'v2 bpc':>9}{'  v1✓':>6}{'  v2✓':>6}")
print("-" * 56)
for a, b in zip(r1, r2):
    print(f"{a['q'][:22]:<24}{a['bpc']:>9.3f}{b['bpc']:>9.3f}{str(a['hit']):>6}{str(b['hit']):>6}")

# 정성 확인 (각 모델 첫 샘플) — 여기 보고 0/1/2 수기 채점
print("\n" + "=" * 60 + "\n정성 샘플 (사람 채점용)\n" + "=" * 60)
for a, b in zip(r1, r2):
    print(f"\nQ: {a['q']}   (정답: {a['a']})")
    print(f"  v1: {a['samples'][0]}")
    print(f"  v2: {b['samples'][0]}")